<h3 style="font-weight:600;color:purple;"> Build Classification Models</h3>

The objective is to predict a *given national cuisine* based on a *group of ingredients*.

<h3 style="font-weight:600;color:purple;">1 | Predict a National Cuisine</h3>

In [1]:
import pandas as pd

cuisines_df = pd.read_csv("../data/cleaned_cuisines.csv")
cuisines_df.head()

,Unnamed: 0,cuisine,almond,angelica,anise,anise_seed,apple,apple_brandy,apricot,armagnac,...,whiskey,white_bread,white_wine,whole_grain_wheat_flour,wine,wood,yam,yeast,yogurt,zucchini
0,0,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,indian,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,3,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,4,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


Import several more libraries:

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
)
from sklearn.svm import SVC
import numpy as np

Divide the X and y coordinates into two dataframes for training. `cuisine` will be the labels dataframe.

In [3]:
cuisines_label_df = cuisines_df["cuisine"]
cuisines_label_df.head()

0    indian
1    indian
2    indian
3    indian
4    indian
Name: cuisine, dtype: str

Drop the `Unnamed: 0` and the `cuisine` columns. The rest of the data will be the trainable features

In [4]:
cuisines_feature_df = cuisines_df.drop(["Unnamed: 0", "cuisine"], axis=1)
cuisines_feature_df.head()

,almond,angelica,anise,anise_seed,apple,apple_brandy,apricot,armagnac,artemisia,artichoke,...,whiskey,white_bread,white_wine,whole_grain_wheat_flour,wine,wood,yam,yeast,yogurt,zucchini
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


<h3 style="font-weight:600;color:purple;">2 | Choosing your Classifier</h3>

<h4 style="font-weight:600;color:purple;">2.1 | What Classifier to go with?</h4>
<h4 style="font-weight:600;color:purple;">2.2 | Reasoning</h4>

- **Neural networks are too heavy**<br/>
Given our clean, but minimal dataset, and the fact that we're running training locally via notebooks, neural networks are too heavy weight for this task.
- **No two-class classifier**<br/>
We do not use a two-class classifier, so that rules out one-vs-all.
- **Multiclass Boosted Decision Trees solve a different problem**<br/>
The multiclass boosted decision tree is most suitable for nonparametric tasks e.g. tasks designed to build rankings, so it is not useful for us.
- **Decision tree or Logistic Regression could work**<br/>
A decision tree might work, or logistic regression for multiclass data.

<h4 style="font-weight:600;color:purple;">2.3 | Using Scikit-learn</h4>

There are two important parameters in Logistic Regression:

- `multi_class`<br/>
Applies a certain behavior.<br/>
If multi_class="ovr", training uses the **one-vs-rest (OvR)** scheme.<br/>
If multi_class="multinomial", training uses the **cross-entropy loss**.
- `solver`<br/>
The algorithm to use in the optimization problem.<br/>
**liblinear** solver does not support multi-class classification (3 or more classes). It is limited to binary (2-class) classification only, so use the default solver **lbfgs**.

<h3 style="font-weight:600;color:purple;">3 | Split the Data</h3>

Split the data into training and testing groups.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(cuisines_feature_df, cuisines_label_df, test_size=0.3)

<h3 style="font-weight:600;color:purple;">4 | Apply Logistic Regression</h3>

1. Create a logistic regression with solver set to the default of `lbfgs` and max_iter set to 1000 to ensure *convergence*. Also note that modern scikit-learn handles multi-class classification automatically (the older `multi_class` parameter is deprecated and no longer supported).

In [6]:
lr = LogisticRegression(solver="lbfgs", max_iter=1000)
model = lr.fit(X_train, y_train)
accuracy = model.score(X_test, y_test)

print(f"Accuracy is {accuracy:.2f}")

Accuracy is 0.82


The accuracy is quite good at **80%!**

2. To see the model in action, test one row of data (index=50) to get the actual values:

In [7]:
print(f"ingredients: {X_test.iloc[50][X_test.iloc[50] != 0].keys()}")
print(f"cuisine: {y_test.iloc[50]}")

ingredients: Index(['basil', 'bean', 'carrot', 'cayenne', 'cilantro', 'cucumber', 'fish',
       'lemongrass', 'lime_juice', 'mint', 'pea', 'peanut', 'scallop',
       'shallot', 'thai_pepper', 'vegetable_oil'],
      dtype='str')
cuisine: thai


3. You can check the accuracy of this prediction:

In [8]:
# By usin double-bracket slicing, we can select our test row (index=50) as a DataFrame, instead of a Series, thus preserving the column names. This is important because the model expects a DataFrame as input for prediction, not a Series.
test = X_test.iloc[[50]]
proba = model.predict_proba(test)
classes = model.classes_
resultdf = pd.DataFrame(data=proba, columns=classes)
topPrediction = resultdf.T.sort_values(by=[0], ascending=False)

topPrediction

,0
thai,0.999266
chinese,0.000510
indian,0.000185
japanese,0.000031
korean,0.000008


So, Korean cuisine is the model's best guess with a probability of **55%**

4. Get more detail by printing a **classification report**:

In [9]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

     chinese       0.71      0.73      0.72       228
      indian       0.94      0.93      0.93       246
    japanese       0.81      0.77      0.79       252
      korean       0.81      0.80      0.81       231
        thai       0.83      0.87      0.85       242

    accuracy                           0.82      1199
   macro avg       0.82      0.82      0.82      1199
weighted avg       0.82      0.82      0.82      1199

